In [1]:
pip install "tensorflow==2.15.*" "tensorflow-hub==0.16.1" "keras==2.15.*" pillow tqdm

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.* (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.*
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tf_keras

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
CLOVE GRADER — EfficientNet-Lite0 Edge Model
=============================================
Fixes the "new images = Not_Clove" problem by:
  1. Using CNN features instead of handcrafted ones (no masking needed)
  2. Heavy domain-gap augmentation for phone photo variation
  3. Label smoothing + Mixup to avoid overfitting on 695 images
  4. INT8 TFLite export for Android (< 5MB, ~30ms inference)

Requires: pip install tensorflow tensorflow_hub pillow tqdm
GPU recommended but CPU works (slower training)
"""

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   
import json
import random
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import tensorflow_hub as hub
import tf_keras as keras
from PIL import Image
from tqdm import tqdm

# ── CONFIG ──────────────────────────────────────────────────────────────────

DATA_ROOT   = Path("/kaggle/input/datasets/patrickiitmz/processed-images-224x224")
COCO_ROOT   = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017")
OUTPUT_DIR  = Path("/kaggle/working/clove_edge_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224          # EfficientNet-Lite0 native size
BATCH_SIZE  = 32
EPOCHS_HEAD = 10           # Train only the classification head first
EPOCHS_FINE = 20           # Fine-tune top layers of the backbone
LR_HEAD     = 1e-3
LR_FINE     = 1e-4
LABEL_SMOOTH = 0.1
MIXUP_ALPHA  = 0.2         # 0 = off
RANDOM_SEED  = 42
N_CALIB_IMGS = 100         # Images for INT8 calibration

CLASS_FOLDERS = {
    "Group_Grade_1": 0,
    "Group_Grade_2": 1,
    "Group_Grade_3": 2,
    "Group_Grade_4": 3,
}
CLASS_NAMES = ['Grade_1', 'Grade_2', 'Grade_3', 'Grade_4', 'Not_Clove']
N_CLASSES   = len(CLASS_NAMES)

# ── EFFICIENTNET-LITE0 URL (TF Hub, no ImageNet top layer) ──────────────────
# Official TF Hub handle — works without internet if cached
EFFNET_URL = "https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2"

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ── DATA LOADING ─────────────────────────────────────────────────────────────

def load_image_paths():
    """Load all image paths + labels. Returns (paths, labels)."""
    paths, labels = [], []

    # Clove grades
    for folder, label in CLASS_FOLDERS.items():
        folder_path = DATA_ROOT / folder
        if not folder_path.exists():
            print(f"  WARNING: {folder_path} not found, skipping")
            continue
        imgs = list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.png"))
        for p in imgs:
            paths.append(str(p))
            labels.append(label)
        print(f"  {folder}: {len(imgs)} images → label {label} ({CLASS_NAMES[label]})")

    # Not_Clove from COCO
    not_clove_paths = []
    for split in ["train2017", "val2017"]:
        sp = COCO_ROOT / split
        if sp.exists():
            not_clove_paths.extend(list(sp.glob("*.jpg")))

    # Balance Not_Clove to ~25% of total
    n_clove = len(paths)
    n_nc = min(len(not_clove_paths), max(200, n_clove // 3))
    chosen = random.sample(not_clove_paths, n_nc)
    for p in chosen:
        paths.append(str(p))
        labels.append(4)
    print(f"  Not_Clove (COCO sample): {n_nc} images → label 4")

    return paths, labels


def split_data(paths, labels, val_ratio=0.15, test_ratio=0.15):
    """Stratified split. Returns (train, val, test) as (paths, labels) tuples."""
    from collections import defaultdict

    class_indices = defaultdict(list)
    for i, label in enumerate(labels):
        class_indices[label].append(i)

    train_idx, val_idx, test_idx = [], [], []
    for label, indices in class_indices.items():
        random.shuffle(indices)
        n = len(indices)
        n_test = max(1, int(n * test_ratio))
        n_val  = max(1, int(n * val_ratio))
        test_idx.extend(indices[:n_test])
        val_idx.extend(indices[n_test:n_test + n_val])
        train_idx.extend(indices[n_test + n_val:])

    def subset(idx):
        p = [paths[i] for i in idx]
        l = [labels[i] for i in idx]
        return p, l

    return subset(train_idx), subset(val_idx), subset(test_idx)


# ── TF DATASET PIPELINE ──────────────────────────────────────────────────────

def decode_image(path, label):
    """Read, decode, resize image to [0,1] float32."""
    raw  = tf.io.read_file(path)
    img  = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32) / 255.0
    return img, label


def augment_train(img, label):
    """
    Heavy augmentation targeting phone-photo domain gap.
    All ops are differentiable and run on the GPU pipeline.
    """
    # Geometric
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)

    # Random 90° rotation (simulates phone orientation)
    k   = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)

    # Color — simulate different phone cameras / lighting
    img = tf.image.random_brightness(img, 0.30)
    img = tf.image.random_contrast(img,   0.70, 1.30)
    img = tf.image.random_saturation(img, 0.60, 1.40)
    img = tf.image.random_hue(img,        0.05)      # slight hue shift

    # JPEG compression artifacts (common on phone photos)
    # Encode + decode to simulate compression, quality 60–95
    quality = tf.random.uniform([], 60, 96, dtype=tf.int32)
    img_uint8 = tf.cast(img * 255.0, tf.uint8)
    img_jpg   = tf.image.adjust_jpeg_quality(img_uint8, quality)
    img       = tf.cast(img_jpg, tf.float32) / 255.0

    # Random crop + resize (simulates different distances)
    crop_size = tf.random.uniform([], 0.75, 1.0)
    h = tf.cast(tf.cast(IMG_SIZE, tf.float32) * crop_size, tf.int32)
    img = tf.image.random_crop(img, [h, h, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # Clip
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label


def mixup(images, labels, alpha=MIXUP_ALPHA):
    """
    Mixup augmentation — interpolates between pairs of images+labels.
    Especially helpful when grade boundaries (Grade_1 vs Grade_2) overlap.
    """
    if alpha == 0:
        return images, labels

    lam = np.random.beta(alpha, alpha, size=(images.shape[0], 1, 1, 1))
    lam = tf.constant(lam, dtype=tf.float32)

    indices  = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    images2  = tf.gather(images,  indices)
    labels2  = tf.gather(labels,  indices)

    mixed_images = lam * images + (1.0 - lam) * images2
    lam_label    = tf.reshape(lam, [-1, 1])
    mixed_labels = lam_label * labels + (1.0 - lam_label) * labels2

    return mixed_images, mixed_labels


def make_dataset(paths, labels, training=False, batch_size=BATCH_SIZE):
    """Build a tf.data pipeline."""
    paths_t  = tf.constant(paths)
    labels_oh = tf.one_hot(labels, N_CLASSES)  # one-hot for label smoothing + mixup

    ds = tf.data.Dataset.from_tensor_slices((paths_t, labels_oh))

    if training:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)

    ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_train, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=training)

    if training and MIXUP_ALPHA > 0:
        ds = ds.map(
            lambda x, y: tf.numpy_function(
                lambda imgs, lbls: mixup(imgs, lbls, MIXUP_ALPHA),
                [x, y], [tf.float32, tf.float32]
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )
        # Restore shapes after numpy_function
        ds = ds.map(
            lambda x, y: (
                tf.ensure_shape(x, [None, IMG_SIZE, IMG_SIZE, 3]),
                tf.ensure_shape(y, [None, N_CLASSES])
            )
        )

    return ds.prefetch(tf.data.AUTOTUNE)


# ── MODEL ────────────────────────────────────────────────────────────────────

def build_model(trainable_backbone=False):
    """
    EfficientNet-Lite0 backbone + classification head.
    Phase 1: backbone frozen  → train only the head.
    Phase 2: unfreeze top 20% of backbone → fine-tune.
    """
    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')

    # TF Hub feature extractor — outputs 1280-dim vector
    backbone = hub.KerasLayer(
        EFFNET_URL,
        trainable=trainable_backbone,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        name='efficientnet_lite0'
    )

    x = backbone(inputs)

    # Classification head
    x = keras.layers.Dropout(0.3, name='dropout')(x)
    x = keras.layers.Dense(256, activation='swish', name='fc1')(x)
    x = keras.layers.BatchNormalization(name='bn1')(x)
    x = keras.layers.Dropout(0.2, name='dropout2')(x)
    outputs = keras.layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs, outputs)
    return model


def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),   # Changed from AdamW
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
        metrics=['accuracy']
    )
    return model


def get_callbacks(name, monitor='val_accuracy'):
    return [
        keras.callbacks.ModelCheckpoint(
            str(OUTPUT_DIR / f'best_{name}.keras'),
            monitor=monitor, save_best_only=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=4,
            min_lr=1e-6, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor=monitor, patience=8,
            restore_best_weights=True, verbose=1
        ),
    ]


# ── TRAINING ─────────────────────────────────────────────────────────────────

def train(train_ds, val_ds):
    print("\n" + "="*60)
    print("PHASE 1 — Training classification head (backbone frozen)")
    print("="*60)

    model = build_model(trainable_backbone=False)
    model = compile_model(model, LR_HEAD)
    model.summary()

    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_HEAD,
        callbacks=get_callbacks('phase1'),
        verbose=1
    )

    # Load best weights
    model.load_weights(str(OUTPUT_DIR / 'best_phase1.keras'))
    
    print("\n" + "="*60)
    print("Skipping backbone fine-tuning (TF1 Hub format limitation)")
    print("Using model with frozen EfficientNet-Lite0 backbone")
    print("="*60)

    return model


# ── EVALUATION ────────────────────────────────────────────────────────────────

def evaluate_model(model, test_ds, test_labels):
    print("\n" + "="*60)
    print("EVALUATION on hold-out test set")
    print("="*60)

    # Predict
    y_pred_probs = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.array(test_labels)

    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

    acc = accuracy_score(y_true, y_pred)
    print(f"\nTest Accuracy: {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Save confusion matrix plot
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f'EfficientNet-Lite0 Clove Grader\nTest Accuracy: {acc:.2%}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
        plt.close()
        print("  Saved confusion_matrix.png")
    except ImportError:
        pass

    return acc


# ── TFLITE EXPORT ─────────────────────────────────────────────────────────────

def get_calibration_dataset(calib_paths):
    """Yield batches of calibration images for INT8 quantization."""
    def representative_dataset():
        random.shuffle(calib_paths)
        for i in range(0, min(N_CALIB_IMGS, len(calib_paths))):
            try:
                img = Image.open(calib_paths[i]).convert('RGB')
                img = img.resize((IMG_SIZE, IMG_SIZE))
                arr = np.array(img, dtype=np.float32) / 255.0
                arr = np.expand_dims(arr, 0)  # [1, H, W, 3]
                yield [arr]
            except Exception:
                continue
    return representative_dataset


def export_tflite(model, calib_paths, variant='int8'):
    """
    Export to TFLite.
    variant: 'int8' (full integer, smallest), 'float16', 'dynamic'
    """
    print(f"\n{'='*60}")
    print(f"EXPORTING TFLite — {variant.upper()}")
    print("="*60)

    # Save as SavedModel first
    saved_model_path = str(OUTPUT_DIR / 'saved_model')
    model.export(saved_model_path)

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)

    if variant == 'int8':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = get_calibration_dataset(calib_paths)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type  = tf.uint8
        converter.inference_output_type = tf.uint8

    elif variant == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif variant == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    out_path = OUTPUT_DIR / f'clove_grader_{variant}.tflite'
    with open(out_path, 'wb') as f:
        f.write(tflite_model)

    size_mb = out_path.stat().st_size / (1024 * 1024)
    print(f"  Saved: {out_path}")
    print(f"  Size:  {size_mb:.2f} MB")
    return out_path


def verify_tflite(tflite_path, test_paths, test_labels):
    """Run a quick accuracy check on the exported TFLite model."""
    print(f"\nVerifying {tflite_path.name}...")

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    inp_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]

    is_int8 = inp_det['dtype'] == np.uint8
    correct  = 0
    total    = 0

    sample = list(zip(test_paths, test_labels))
    random.shuffle(sample)
    sample = sample[:200]  # check on 200 images

    for path, label in sample:
        try:
            img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(img, dtype=np.float32) / 255.0

            if is_int8:
                scale, zp = inp_det['quantization']
                arr = (arr / scale + zp).clip(0, 255).astype(np.uint8)

            interpreter.set_tensor(inp_det['index'], arr[np.newaxis])
            interpreter.invoke()
            out = interpreter.get_tensor(out_det['index'])[0]

            if is_int8:
                scale, zp = out_det['quantization']
                out = (out.astype(np.float32) - zp) * scale

            pred = int(np.argmax(out))
            if pred == label:
                correct += 1
            total += 1
        except Exception:
            continue

    acc = correct / max(total, 1)
    print(f"  TFLite accuracy on {total} samples: {acc:.4f} ({acc*100:.2f}%)")
    return acc


# ── FLUTTER/ANDROID INTEGRATION CODE ──────────────────────────────────────────

FLUTTER_INFERENCE_CODE = '''
// ============================================================
// Flutter: Clove grading with EfficientNet-Lite0 INT8 TFLite
// ============================================================
// pubspec.yaml dependencies:
//   tflite_flutter: ^0.10.4
//   image: ^4.0.17

import 'dart:typed_data';
import 'package:tflite_flutter/tflite_flutter.dart';
import 'package:image/image.dart' as img;

class CloveGrader {
  static const int imgSize = 224;
  static const List<String> labels = [
    'Grade_1', 'Grade_2', 'Grade_3', 'Grade_4', 'Not_Clove'
  ];

  late Interpreter _interpreter;
  bool _isInt8 = false;

  Future<void> load() async {
    // Load INT8 model from assets
    _interpreter = await Interpreter.fromAsset(
      'assets/clove_grader_int8.tflite',
      options: InterpreterOptions()..threads = 4,
    );

    // Detect if model uses INT8 or FLOAT32 input
    final inputType = _interpreter.getInputTensor(0).type;
    _isInt8 = (inputType == TfLiteType.uint8);

    _interpreter.allocateTensors();
  }

  Map<String, dynamic> grade(img.Image rawImage) {
    // Resize to 224x224
    final resized = img.copyResize(rawImage, width: imgSize, height: imgSize);

    if (_isInt8) {
      // INT8 model: pixels already 0-255 as uint8
      final input = List.generate(
        1, (_) => List.generate(imgSize, (y) =>
          List.generate(imgSize, (x) {
            final pixel = resized.getPixel(x, y);
            return [pixel.r.toInt(), pixel.g.toInt(), pixel.b.toInt()];
          })
        )
      );

      final output = List.filled(1 * labels.length, 0).reshape([1, labels.length]);
      _interpreter.run(input, output);

      // Dequantize output (get quantization params from model metadata)
      // For simplicity, find argmax directly on raw uint8 output
      final scores = (output[0] as List).cast<int>();
      final maxIdx = scores.indexOf(scores.reduce((a, b) => a > b ? a : b));
      return {
        'grade': labels[maxIdx],
        'confidence': scores[maxIdx] / 255.0,
        'is_clove': maxIdx != 4,
        'action': _getAction(labels[maxIdx]),
      };
    } else {
      // FLOAT32 model: normalize to [0, 1]
      final input = List.generate(
        1, (_) => List.generate(imgSize, (y) =>
          List.generate(imgSize, (x) {
            final pixel = resized.getPixel(x, y);
            return [pixel.r / 255.0, pixel.g / 255.0, pixel.b / 255.0];
          })
        )
      );

      final output = [List.filled(labels.length, 0.0)];
      _interpreter.run(input, output);

      final scores = output[0];
      final maxIdx = scores.indexOf(scores.reduce((a, b) => a > b ? a : b));
      return {
        'grade': labels[maxIdx],
        'confidence': scores[maxIdx],
        'is_clove': maxIdx != 4,
        'action': _getAction(labels[maxIdx]),
      };
    }
  }

  String _getAction(String grade) {
    const actions = {
      'Grade_1': 'EXPORT READY — Premium quality',
      'Grade_2': 'EXPORT — Standard quality',
      'Grade_3': 'REPROCESS — Consider local sale',
      'Grade_4': 'REJECT — Khoker, do not mix',
      'Not_Clove': 'INVALID — Retake photo',
    };
    return actions[grade] ?? 'Unknown';
  }

  void dispose() => _interpreter.close();
}
'''


def save_android_assets(output_dir):
    """Save all files needed for Android/Flutter integration."""
    # Class names JSON
    with open(output_dir / 'class_names.json', 'w') as f:
        json.dump(CLASS_NAMES, f, indent=2)

    # Model metadata
    with open(output_dir / 'model_config.json', 'w') as f:
        json.dump({
            'model': 'EfficientNet-Lite0',
            'input_size': IMG_SIZE,
            'n_classes': N_CLASSES,
            'class_names': CLASS_NAMES,
            'input_range': '0-255 (uint8) for INT8 model, 0.0-1.0 (float32) for others',
            'preprocessing': 'resize to 224x224, no mean subtraction needed',
        }, f, indent=2)

    # Flutter inference code
    with open(output_dir / 'flutter_inference.dart', 'w') as f:
        f.write(FLUTTER_INFERENCE_CODE)

    print("\nAndroid/Flutter assets saved:")
    for p in sorted(output_dir.iterdir()):
        size = p.stat().st_size / 1024
        print(f"  {p.name:45} {size:8.1f} KB")


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    print("="*60)
    print("CLOVE GRADER — EfficientNet-Lite0 Edge Model")
    print("="*60)
    print(f"TensorFlow version: {tf.__version__}")
    print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

    # 1. Load data
    print("\n[1/5] Loading data...")
    paths, labels = load_image_paths()
    print(f"\n  Total images: {len(paths)}")
    for i, name in enumerate(CLASS_NAMES):
        count = labels.count(i)
        print(f"  {name:12}: {count:5} ({count/len(paths)*100:.1f}%)")

    (train_paths, train_labels), \
    (val_paths,   val_labels),   \
    (test_paths,  test_labels)   = split_data(paths, labels)

    print(f"\n  Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

    # 2. Build datasets
    print("\n[2/5] Building tf.data pipelines...")
    train_ds = make_dataset(train_paths, train_labels, training=True)
    val_ds   = make_dataset(val_paths,   val_labels,   training=False)
    test_ds  = make_dataset(test_paths,  test_labels,  training=False)

    # 3. Train
    print("\n[3/5] Training...")
    model = train(train_ds, val_ds)

    # 4. Evaluate
    print("\n[4/5] Evaluating...")
    evaluate_model(model, test_ds, test_labels)

    # 5. Export TFLite
    print("\n[5/5] Exporting TFLite models...")

    # Use training images for calibration (not test)
    calib_paths = [p for p in train_paths if not p.endswith('coco')]
    random.shuffle(calib_paths)
    calib_paths = calib_paths[:N_CALIB_IMGS]

    # Export all three variants so you can compare
    tflite_int8    = export_tflite(model, calib_paths, 'int8')
    tflite_fp16    = export_tflite(model, calib_paths, 'float16')
    tflite_dynamic = export_tflite(model, calib_paths, 'dynamic')

    # Verify exported models
    verify_tflite(tflite_int8,    test_paths, test_labels)
    verify_tflite(tflite_fp16,    test_paths, test_labels)
    verify_tflite(tflite_dynamic, test_paths, test_labels)

    # Save Flutter integration code
    save_android_assets(OUTPUT_DIR)

    print("\n" + "="*60)
    print("DONE")
    print("="*60)
    print(f"\nFiles in {OUTPUT_DIR}:")
    for p in sorted(OUTPUT_DIR.iterdir()):
        print(f"  {p.name}")


if __name__ == '__main__':
    main()

2026-04-28 12:06:54.391695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777378014.636369      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777378014.712154      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777378015.275006      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777378015.275048      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777378015.275051      23 computation_placer.cc:177] computation placer alr

CLOVE GRADER — EfficientNet-Lite0 Edge Model
TensorFlow version: 2.19.0
GPU available: True

[1/5] Loading data...
  Group_Grade_1: 172 images → label 0 (Grade_1)
  Group_Grade_2: 210 images → label 1 (Grade_2)
  Group_Grade_3: 156 images → label 2 (Grade_3)
  Group_Grade_4: 157 images → label 3 (Grade_4)
  Not_Clove (COCO sample): 231 images → label 4

  Total images: 926
  Grade_1     :   172 (18.6%)
  Grade_2     :   210 (22.7%)
  Grade_3     :   156 (16.8%)
  Grade_4     :   157 (17.0%)
  Not_Clove   :   231 (24.9%)

  Train: 654, Val: 136, Test: 136

[2/5] Building tf.data pipelines...


I0000 00:00:1777378045.185765      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777378045.191816      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



[3/5] Training...

PHASE 1 — Training classification head (backbone frozen)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 224, 224, 3)]     0         
                                                                 
 efficientnet_lite0 (KerasL  (None, 1280)              3413024   
 ayer)                                                           
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 fc1 (Dense)                 (None, 256)               327936    
                                                                 
 bn1 (BatchNormalization)    (None, 256)               1024      
                                                                 
 dropout2 (Dropout)          (None, 256)          

I0000 00:00:1777378055.487641      76 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777378058.332405      73 service.cc:152] XLA service 0x7982d930eda0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777378058.332442      73 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777378058.332446      73 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777378058.612814      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

  Saved: /kaggle/working/clove_edge_model/clove_grader_float16.tflite
  Size:  7.06 MB

EXPORTING TFLite — DYNAMIC
INFO:tensorflow:Assets written to: /kaggle/working/clove_edge_model/saved_model/assets


INFO:tensorflow:Assets written to: /kaggle/working/clove_edge_model/saved_model/assets


Saved artifact at '/kaggle/working/clove_edge_model/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  133604036226384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036226000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036226960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036226576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036227344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036225040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036227536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036227152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036227728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133604036228112: TensorSpec(shape=(), dtype=tf.resource, 

W0000 00:00:1777378107.075097      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1777378107.075158      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


  Saved: /kaggle/working/clove_edge_model/clove_grader_dynamic.tflite
  Size:  3.85 MB

Verifying clove_grader_int8.tflite...
  TFLite accuracy on 136 samples: 0.8015 (80.15%)

Verifying clove_grader_float16.tflite...
  TFLite accuracy on 136 samples: 0.8162 (81.62%)

Verifying clove_grader_dynamic.tflite...
  TFLite accuracy on 136 samples: 0.8088 (80.88%)

Android/Flutter assets saved:
  best_phase1.keras                              17320.4 KB
  class_names.json                                   0.1 KB
  clove_grader_dynamic.tflite                     3938.7 KB
  clove_grader_float16.tflite                     7225.1 KB
  clove_grader_int8.tflite                        4186.5 KB
  confusion_matrix.png                              55.4 KB
  flutter_inference.dart                             3.2 KB
  model_config.json                                  0.3 KB
  saved_model                                        4.0 KB

DONE

Files in /kaggle/working/clove_edge_model:
  best_phase1.keras